# Hybrid Low-Resource TTS Dataset Synthesis Pipeline

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Turns real ASR recordings with human transcripts, such as [Common Voice](https://commonvoice.mozilla.org/en/datasets), into a single-voice TTS training set: format conversion, denoising, trimming, normalization, WPM filtering, voice conversion to one target speaker and optional punctuation restoration. The converted set is what [`train_vits.ipynb`](train_vits.ipynb) trains a VITS voice on. Denoising uses [audiosronnx](https://github.com/TigreGotico/audiosronnx) and voice conversion uses [voiceclonnx](https://github.com/TigreGotico/voiceclonnx); both run on `onnxruntime` alone, so no PyTorch or CUDA driver is needed for either stage.

Three quality gates measure the data with [speechonnxmetrics](https://github.com/TigreGotico/speechonnxmetrics) and drop the clips that fail. Every clip's scores are written to `quality_scores.csv`, and a summary at the end shows what each gate removed.

| Gate | Where | Measures | Default threshold |
|---|---|---|---|
| 1. Input quality | after denoising (Step 2) | DNSMOS overall (and NISQA MOS when selected) of the clip before and after the denoiser | `MIN_INPUT_DNSMOS_OVRL=2.5` |
| 2. Transcript | after the WPM filter (Step 5) | CER between the given transcript and an ASR round trip on the cleaned clip | `MAX_CER=0.25` |
| 3. Converted voice | after voice conversion (Step 6) | speaker similarity to the target reference voice, and UTMOS | `MIN_SPK_COS=0.3`, `MIN_UTMOS=3.0` |

| Field | Value |
|---|---|
| Inputs | a Common Voice language directory (`COMMON_VOICE_DIR`: `clips/` and `validated.tsv`), or any raw donor audio directory (`RAW_AUDIO_SOURCE`) with its transcript (`METADATA_CSV`); and one reference WAV per target voice (`REF_VOICES_DIR`). With none of them set, the notebook runs a demo on a small public Common Voice slice |
| Outputs | a voice-converted LJSpeech-style dataset, a `manifest.jsonl` ([manifest.md](../docs/manifest.md)) carrying the gate scores in each row's `scores`, `quality_scores.csv` (one row per clip) and `gate_summary.json`, all under `WORK_DIR` |
| Runtime | demo (default): about 15 to 30 minutes on CPU, most of it model downloads on the first run; full run: minutes to tens of minutes depending on dataset size and the chosen `DENOISE_ENGINE`/`VC_ENGINE`; CPU smoke path (`SMOKE=True`, `N_SAMPLES` files): a few minutes, with the transcript gate and punctuation restoration skipped |
| Hardware | CPU (`onnxruntime`); `ONNX_RUNTIME=cuda` or `rocm` speeds up denoising, voice conversion and the gates on their own hardware, but neither needs one |
| Cost | CPU. The demo downloads about 1.2 GB of models (ASR 670 MB, Chatterbox INT8 468 MB, UTMOS 411 MB, plus small denoiser, DNSMOS and speaker models) and about 30 MB of audio from Hugging Face; no token is needed |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path.

### Quick start

Press Run All with nothing configured. The notebook prints that it is running the demo, fetches a few validated Common Voice clips and a public reference voice, and exercises every gate (one demo transcript is replaced on purpose so the transcript gate has a clip to catch). To run on your data, set `COMMON_VOICE_DIR` and `REF_VOICES_DIR` (Common Voice) or `RAW_AUDIO_SOURCE`, `METADATA_CSV` and `REF_VOICES_DIR` (any other corpus) before the configuration cell.

The demo data are Common Voice 17.0 (Mozilla Foundation, CC0 1.0) and one LibriSpeech utterance (Panayotov et al., CC BY 4.0, from LibriVox public-domain recordings); Step 0 gives the full attribution.

### Common Voice as input

Download a language from [Common Voice](https://commonvoice.mozilla.org/en/datasets) and extract it. The language directory holds `clips/*.mp3` and `validated.tsv` with the columns `client_id`, `path`, `sentence`, `up_votes` and `down_votes`. Step 0 reads that TSV and builds the notebook's two inputs from it:

- `CV_MIN_UP_VOTES` and `CV_MAX_DOWN_VOTES` keep clips by their votes (default 2 and 1).
- `CV_SPEAKER` keeps one speaker: `auto` picks the `client_id` with the most hours among the three with the most clips, or give a `client_id` (a unique prefix is enough). Empty keeps everyone. A single speaker gives a more consistent training set; many speakers give more hours.
- `CV_MAX_CLIPS` caps the number of clips (0 = no cap).
- `RAW_AUDIO_SOURCE` becomes a directory of links to the selected clips and `METADATA_CSV` a `file|sentence` file, so every later step runs unchanged.

Set `TTS_LANG` to the BCP-47 tag of the language (for example `pt-PT`); it picks the ASR model for the transcript gate.

**Read more:**

- [OVOS TTS plugins](https://openvoiceos.github.io/ovos-technical-manual/320-tts_plugins/)
- [phoonnx](https://github.com/TigreGotico/phoonnx)
- [voiceclonnx](https://github.com/TigreGotico/voiceclonnx)
- [audiosronnx](https://github.com/TigreGotico/audiosronnx)
- [speechonnxmetrics](https://github.com/TigreGotico/speechonnxmetrics)
- [Making synthetic voices from scratch](https://openvoiceos.github.io/ovos-blogs/posts/2025-06-26-making-synthetic-voices-from-scratch)

In [ ]:
SMOKE = False
N_SAMPLES = 32
# EPOCHS does not apply: no training happens in this notebook.


In [ ]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return os.path.abspath(f"./{name}")


def _env(key, default=""):
    return os.environ.get(key, default)


def _parse_typed_env(env_name, raw, kind):
    if kind == "bool":
        low = raw.strip().lower()
        if low in ("1", "true", "yes", "on"):
            return True
        if low in ("0", "false", "no", "off"):
            return False
        raise ValueError(f"{env_name}={raw!r} is not a valid bool (use true/false)")
    if kind == "int":
        try:
            return int(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid int") from None
    if kind == "float":
        try:
            return float(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid float") from None
    return raw


def _resolve_onnx_providers(raw):
    """A comma list, or "auto": CUDA, ROCm, MIGraphX, CPU, CPU always last."""
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort

        preferred = [
            "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider",
            "CPUExecutionProvider",
        ]
        available = ort.get_available_providers()
        resolved = [p for p in preferred if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


# ==========================================
# GLOBAL CONFIGURATION (ENV VARS)
# ==========================================

# Values this notebook derived on an earlier run in the same kernel (the SMOKE fixtures, the
# demo inputs, defaults that depend on SMOKE) are dropped here and derived again, so a second
# Run All, or a SMOKE run followed by a normal one, never inherits them. A value the user set is
# never recorded and never dropped.
_DERIVED = globals().get("_DERIVED", {})
for _key, _value in list(_DERIVED.items()):
    if os.environ.get(_key) == _value:
        del os.environ[_key]
_DERIVED.clear()


def _derive(key, value):
    os.environ[key] = value
    _DERIVED[key] = value


def _default(key, value):
    if key not in os.environ:
        _derive(key, value)


os.environ.setdefault("WORK_DIR", _default_base_dir("asr2tts_workspace"))
os.makedirs(os.environ["WORK_DIR"], exist_ok=True)

# Input data sources. Three ways in, checked in this order:
#   1. SMOKE=True builds a tiny fixture set of sine tones (CI only).
#   2. Every one of the variables below left empty runs the DEMO: a small
#      anonymous Common Voice slice and a public reference voice from Hugging Face.
#   3. A real run: COMMON_VOICE_DIR (a Common Voice language directory), or
#      RAW_AUDIO_SOURCE + METADATA_CSV for any other donor corpus, plus REF_VOICES_DIR.
os.environ.setdefault("RAW_AUDIO_SOURCE", "")
os.environ.setdefault("METADATA_CSV", "")
os.environ.setdefault("REF_VOICES_DIR", "")

# Common Voice as input (Step 0). COMMON_VOICE_DIR holds the extracted language
# directory of a Common Voice release: `clips/` and `validated.tsv`. Step 0 selects
# clips from the TSV and fills RAW_AUDIO_SOURCE and METADATA_CSV from them.
os.environ.setdefault("COMMON_VOICE_DIR", "")
os.environ.setdefault("CV_TSV", "validated.tsv")
# "" = every speaker; "auto" = the client_id with the most hours among the three
# with the most clips; anything else = that client_id (a unique prefix is enough).
os.environ.setdefault("CV_SPEAKER", "")
os.environ.setdefault("CV_MIN_UP_VOTES", "2")     # validated.tsv already guarantees 2 up votes
os.environ.setdefault("CV_MAX_DOWN_VOTES", "1")   # a clip with more down votes is dropped
os.environ.setdefault("CV_MAX_CLIPS", "0")        # 0 = no cap

# The demo (used when no input variable is set): a few validated clips from a public
# Hugging Face slice of Common Voice (CC0) and one LibriSpeech utterance as the target
# voice (CC BY 4.0). Both load without a token.
os.environ.setdefault("DEMO_CLIPS", "12")
os.environ.setdefault("DEMO_DATASET", "fixie-ai/common_voice_17_0")
os.environ.setdefault("DEMO_CONFIG", "en")
os.environ.setdefault("DEMO_SPLIT", "validation")
os.environ.setdefault("DEMO_REF_DATASET", "hf-internal-testing/librispeech_asr_dummy")
os.environ.setdefault("DEMO_REF_CONFIG", "clean")
os.environ.setdefault("DEMO_REF_SPLIT", "validation")
os.environ.setdefault("DEMO_REF_ROW", "2")

# The BCP-47 tag written into manifest.jsonl's "lang" field (docs/manifest.md) and used
# to pick an ASR model for the transcript gate.
os.environ.setdefault("TTS_LANG", "en-US")

# Audio denoising (Step 2, audiosronnx load_denoise). audiosronnx.available_denoisers()
# prints every alias; the default, gtcrn, is a 0.54 MB model (audiosronnx's README) chosen
# so the smoke path never downloads a large file.
os.environ.setdefault("DENOISE_ENGINE", "gtcrn")

# Voice conversion (Step 6, voiceclonnx). `voiceclonnx list` prints every alias; the
# default, chatterbox, is an audio-to-audio cloner with natural prosody and one of the
# strongest source-to-target voice shifts (voiceclonnx's README engine comparison); its
# INT8 weights are about 468 MB. SMOKE uses the tiny triaan engine to stay cheap.
_default("VC_ENGINE", "triaan" if SMOKE else "chatterbox")
os.environ.setdefault("VC_QUANTIZED", "true")

# onnxruntime, shared by audiosronnx and voiceclonnx.
os.environ.setdefault("ONNX_RUNTIME", "cpu")     # cpu|cuda|rocm - which onnxruntime build the install cells fetch
os.environ.setdefault("ONNX_PROVIDERS", "auto")  # execution providers: a comma list, or "auto"

# Punctuation restoration (Step 7): an onnxruntime checkpoint, PUNCT_MODEL_SUBDIR
# naming the subdirectory of PUNCT_MODEL that holds its onnx export.
os.environ.setdefault("PUNCT_MODEL", "oliverguhr/fullstop-punctuation-multilang-large")
os.environ.setdefault("PUNCT_MODEL_SUBDIR", "onnx")

# Processing parameters (Steps 3-5)
os.environ.setdefault("TARGET_LUFS", "-20.0")
os.environ.setdefault("WPM_STD_THRESHOLD", "2.0")
os.environ.setdefault("MIN_TRIM_DURATION", "1.0")

# ==========================================
# QUALITY GATES (speechonnxmetrics)
# ==========================================
# ENFORCE_GATES=false only scores: every clip is kept and the table still says which
# clips would have been dropped. SMOKE enforces too, with thresholds its sine-tone fixtures
# pass and one planted clip that must fail Gate 1, so CI runs the dropping path.
_default("ENFORCE_GATES", "true")

# Gate 1, after denoising: no-reference MOS of the denoised clip. A comma list of
# `dnsmos` (MIT weights) and `nisqa` (CC BY-NC-SA weights: non-commercial use only).
# Both are scored 1-5; 2.5 is the middle of the poor-to-fair band, so a clip under it
# is one that a listener would call noisy or distorted even after denoising.
os.environ.setdefault("QUALITY_INPUT_METRICS", "dnsmos")
_default("MIN_INPUT_DNSMOS_OVRL", "1.7" if SMOKE else "2.5")  # SMOKE: tones score 1.9, the planted square wave 1.5 after denoising
os.environ.setdefault("MIN_INPUT_NISQA_MOS", "2.5")

# Gate 2, after the WPM filter: ASR round trip on the cleaned clip against its given
# transcript. ASR_MODEL names an onnx-asr model (empty = the default multilingual
# model when it covers TTS_LANG); ASR_STT_MODULE names an OVOS STT plugin instead.
# No Whisper model is accepted. MAX_CER 0.25: a correct transcript with a modern
# recogniser lands near 0.05-0.10 even with accents and noise, a different sentence at
# 0.5 or more, so 0.25 removes mismatches and keeps hard-but-correct clips.
os.environ.setdefault("ASR_MODEL", "")
os.environ.setdefault("ASR_QUANTIZATION", "int8")
os.environ.setdefault("ASR_STT_MODULE", "")
os.environ.setdefault("ASR_STT_CONFIG", "{}")
os.environ.setdefault("MAX_CER", "0.25")

# Gate 3, after voice conversion: speaker similarity of the converted clip to the target
# reference (speakeronnx wespeaker-resnet34 cosine, the `spk_cos` of tts_measure) and its
# UTMOS (1-5, natural speech scores above about 3.5). MIN_SPK_COS 0.3 sits above the
# 0-0.2 band where different speakers fall and below the 0.5 or more of one speaker;
# MIN_UTMOS 3.0 is the "fair" point of the 1-5 scale.
_default("MIN_SPK_COS", "0.3")  # SMOKE sets a threshold per voice from its own scores (Gate 3 cell)
os.environ.setdefault("MIN_UTMOS", "3.0")
# UTMOS weights are about 411 MB; SMOKE skips them.
_default("SCORE_UTMOS", "false" if SMOKE else "true")

# Punctuation restoration (Step 7) downloads over 1 GB. Common Voice sentences are already
# punctuated, so it defaults to off for them, for the demo and for SMOKE.
_default(
    "RESTORE_PUNCTUATION",
    "false" if (SMOKE or os.environ["COMMON_VOICE_DIR"] or not (
        os.environ["RAW_AUDIO_SOURCE"] or os.environ["METADATA_CSV"] or os.environ["REF_VOICES_DIR"]
    )) else "true",
)
if SMOKE:
    # Generate a handful of short sine-tone WAV fixtures so the smoke path has
    # something to convert, denoise, trim and voice-convert, without a real
    # donor dataset or reference voices.
    import array
    import math
    import wave

    sample_rate = 16000

    def _write_tone(path, freq, seconds):
        n = sample_rate * seconds
        samples = array.array("h", [
            int(3000 * math.sin(2 * math.pi * freq * t / sample_rate)) for t in range(n)
        ])
        with wave.open(path, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(samples.tobytes())

    smoke_src = os.path.join(os.environ["WORK_DIR"], "smoke_raw_audio")
    os.makedirs(smoke_src, exist_ok=True)
    for i in range(N_SAMPLES):
        _write_tone(os.path.join(smoke_src, f"sample_{i:02d}.wav"), 220, 2)
    # A loud 3 kHz square wave: Gate 1 must drop it (DNSMOS overall about 1.1 after denoising,
    # against 1.9 for the tones), which proves in CI that the gate can fail and that a failing
    # clip moves to rejected/.
    with wave.open(os.path.join(smoke_src, "sample_bad.wav"), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(array.array("h", [
            29000 if (t * 6000 // sample_rate) % 2 == 0 else -29000 for t in range(sample_rate * 2)
        ]).tobytes())
    _derive("RAW_AUDIO_SOURCE", smoke_src)

    smoke_csv = os.path.join(os.environ["WORK_DIR"], "smoke_metadata.csv")
    with open(smoke_csv, "w") as f:
        for i in range(N_SAMPLES):
            f.write(f"sample_{i:02d}|this is smoke fixture sentence {i}\n")
        f.write("sample_bad|this clip is the planted failure\n")
    _derive("METADATA_CSV", smoke_csv)

    smoke_refs = os.path.join(os.environ["WORK_DIR"], "smoke_ref_voices")
    os.makedirs(smoke_refs, exist_ok=True)
    for name, freq in (("ref_a", 180), ("ref_b", 260)):
        _write_tone(os.path.join(smoke_refs, f"{name}.wav"), freq, 1)
    _derive("REF_VOICES_DIR", smoke_refs)

# Define pipeline subdirectories (chain of custody)
base = os.environ["WORK_DIR"]
os.environ["DIR_CONVERTED"] = os.path.join(base, "1_converted")
os.environ["DIR_DENOISED"] = os.path.join(base, "2_denoised")
os.environ["DIR_TRIMMED"] = os.path.join(base, "3_trimmed")
os.environ["DIR_NORMALIZED"] = os.path.join(base, "4_normalized")
os.environ["DIR_FILTERED"] = os.path.join(base, "5_wpm_filtered")
os.environ["DIR_FINAL_VC"] = os.path.join(base, "6_vc_output")

print(f"Workspace: {os.environ['WORK_DIR']}")


---
### Step 0: Input

Resolves the three inputs. With `SMOKE=True` it uses the sine-tone fixtures. With every input variable empty it runs the demo: it downloads a few validated Common Voice clips and one LibriSpeech utterance (the target voice) from Hugging Face, writes them as a Common Voice language directory and continues as a Common Voice run. With `COMMON_VOICE_DIR` set it maps that directory's `validated.tsv` onto `RAW_AUDIO_SOURCE` and `METADATA_CSV` (see the table at the top). Otherwise `RAW_AUDIO_SOURCE`, `METADATA_CSV` and `REF_VOICES_DIR` must all be set.

Demo data and licences: the clips are from [Common Voice 17.0](https://commonvoice.mozilla.org) (Mozilla Foundation and its contributors, CC0 1.0), read through the Hugging Face mirror `fixie-ai/common_voice_17_0`. The target voice is one utterance of speaker 1272 from [LibriSpeech](https://www.openslr.org/12) (V. Panayotov, G. Chen, D. Povey and S. Khudanpur, ICASSP 2015, built from LibriVox public-domain recordings, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)), read through `hf-internal-testing/librispeech_asr_dummy`. Neither Hugging Face repository states a licence of its own; the terms above are the upstream corpora's. The demo reads the datasets through the public Hugging Face dataset viewer, so it depends on those two repositories staying available.

In [ ]:
%pip install --quiet --pre "soundfile>=0.14.0"

import csv
import io
import json
import re
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request

import soundfile as sf

INPUT_VARIABLES = ("RAW_AUDIO_SOURCE", "METADATA_CSV", "REF_VOICES_DIR", "COMMON_VOICE_DIR")
DEMO = not SMOKE and not any(_env(k) for k in INPUT_VARIABLES)
WORK = _env("WORK_DIR")


def _http_get(url, tries=3):
    """GET with a short backoff: the dataset viewer rate-limits bursts and a read can stall."""
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "asr2tts"}), timeout=60) as r:
                return r.read()
        except OSError as exc:  # URLError, TimeoutError, ConnectionError
            if attempt == tries - 1:
                raise RuntimeError(f"could not fetch {url.split('?')[0]}: {exc}") from exc
            time.sleep(2 ** attempt)


def _viewer_rows(dataset, config, split, offset, length):
    query = urllib.parse.urlencode(
        {"dataset": dataset, "config": config, "split": split, "offset": offset, "length": length}
    )
    return json.loads(_http_get(f"https://datasets-server.huggingface.co/rows?{query}"))["rows"]


def _save_wav(row, path):
    """The viewer may serve FLAC or WAV; write 16-bit PCM WAV either way."""
    if not os.path.exists(path):
        data, rate = sf.read(io.BytesIO(_http_get(row["audio"][0]["src"])))
        sf.write(path, data, rate, subtype="PCM_16")


def build_demo(demo_root):
    """Write a Common Voice language directory (clips/ + validated.tsv) and one reference voice."""
    cv_dir = os.path.join(demo_root, "common_voice_demo")
    ref_dir = os.path.join(demo_root, "ref_voices")
    os.makedirs(os.path.join(cv_dir, "clips"), exist_ok=True)
    os.makedirs(ref_dir, exist_ok=True)

    n_clips = int(_env("DEMO_CLIPS"))
    rows = _viewer_rows(_env("DEMO_DATASET"), _env("DEMO_CONFIG"), _env("DEMO_SPLIT"), 0, n_clips)
    planted = None
    with open(os.path.join(cv_dir, "validated.tsv"), "w", newline="", encoding="utf-8") as f:
        out = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        out.writerow(["client_id", "path", "sentence", "up_votes", "down_votes"])
        for i, item in enumerate(rows):
            row = item["row"]
            # The viewer serves wav; the TSV names the wav, where a real release names mp3.
            name = os.path.splitext(os.path.basename(row["path"]))[0] + ".wav"
            _save_wav(row, os.path.join(cv_dir, "clips", name))
            sentence = row["sentence"]
            if i == 1:
                # One planted wrong transcript, so the transcript gate has something to catch.
                planted, sentence = name, "The quick brown fox jumps over the lazy dog."
            out.writerow([row["client_id"], name, sentence, row["up_votes"], row["down_votes"]])

    ref_row = _viewer_rows(
        _env("DEMO_REF_DATASET"), _env("DEMO_REF_CONFIG"), _env("DEMO_REF_SPLIT"),
        int(_env("DEMO_REF_ROW")), 1,
    )[0]["row"]
    _save_wav({"audio": ref_row["audio"]}, os.path.join(ref_dir, "ref_librispeech.wav"))
    return cv_dir, ref_dir, planted


def _wav_hours(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate / 3600
    except Exception:
        return 0.0


def select_common_voice(cv_dir, out_dir, out_csv):
    """Map a Common Voice language directory onto RAW_AUDIO_SOURCE (a directory of
    links to the selected clips) and METADATA_CSV (`stem|sentence`)."""
    tsv_path = os.path.join(cv_dir, _env("CV_TSV"))
    clips_dir = os.path.join(cv_dir, "clips")
    if not os.path.isfile(tsv_path) or not os.path.isdir(clips_dir):
        raise FileNotFoundError(
            f"COMMON_VOICE_DIR={cv_dir} must hold {_env('CV_TSV')} and a clips/ directory "
            f"(the extracted language folder of a Common Voice release, for example cv-corpus-*/en)."
        )
    min_up, max_down = int(_env("CV_MIN_UP_VOTES")), int(_env("CV_MAX_DOWN_VOTES"))
    with open(tsv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
        rows = [r for r in reader if int(r.get("up_votes") or 0) >= min_up and int(r.get("down_votes") or 0) <= max_down]
    print(f"{tsv_path}: {len(rows)} clip(s) with up_votes >= {min_up} and down_votes <= {max_down}")

    speaker = _env("CV_SPEAKER").strip()
    if speaker:
        by_client = {}
        for r in rows:
            by_client.setdefault(r["client_id"], []).append(r)
        if speaker == "auto":
            top = sorted(by_client, key=lambda c: len(by_client[c]), reverse=True)[:3]
            hours = {}
            for client in top:
                sample = by_client[client][:200]
                known = [_wav_hours(os.path.join(clips_dir, r["path"])) for r in sample]
                hours[client] = sum(known) / max(len(sample), 1) * len(by_client[client])
                print(f"  speaker {client[:12]}: {len(by_client[client])} clips, about {hours[client]:.2f} h")
            speaker = max(hours, key=hours.get)
        else:
            matches = [c for c in by_client if c.startswith(speaker)]
            if len(matches) != 1:
                raise RuntimeError(f"CV_SPEAKER={speaker!r} matches {len(matches)} client_id values; give a unique prefix")
            speaker = matches[0]
        rows = by_client[speaker]
        print(f"Single speaker: client_id {speaker[:12]}..., {len(rows)} clip(s)")

    cap = int(_env("CV_MAX_CLIPS"))
    if cap > 0:
        rows = rows[:cap]
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)  # links from an earlier selection must not leak into this one
    os.makedirs(out_dir, exist_ok=True)
    kept = []
    for r in rows:
        src = os.path.join(clips_dir, r["path"])
        if not os.path.isfile(src):
            continue
        stem, ext = os.path.splitext(r["path"])
        dst = os.path.join(out_dir, stem + ext)
        if not os.path.lexists(dst):
            try:
                os.symlink(os.path.abspath(src), dst)
            except OSError:
                shutil.copy2(src, dst)
        # The metadata file is pipe-separated and unquoted.
        text = re.sub(r"[|\t\r\n]+", " ", r["sentence"]).strip()
        kept.append(f"{stem}|{text}")
    with open(out_csv, "w", encoding="utf-8") as f:
        f.write("\n".join(kept))
    print(f"Selected {len(kept)} Common Voice clip(s) -> {out_dir}")


if SMOKE:
    print("SMOKE=True: using the sine-tone fixtures built above.")
elif DEMO:
    print("=" * 78)
    print("DEMO RUN: no input variable is set, so this run uses a public demo slice:")
    print(f"  {_env('DEMO_CLIPS')} validated Common Voice clips from {_env('DEMO_DATASET')} "
          f"({_env('DEMO_CONFIG')}/{_env('DEMO_SPLIT')}) and one LibriSpeech utterance as the target voice.")
    print("  To run on your data, set COMMON_VOICE_DIR (a Common Voice language directory) or")
    print("  RAW_AUDIO_SOURCE + METADATA_CSV (any corpus), and REF_VOICES_DIR (one WAV per target voice).")
    print("=" * 78)
    cv_dir, ref_dir, planted = build_demo(os.path.join(WORK, "demo_input"))
    _derive("COMMON_VOICE_DIR", cv_dir)
    _derive("REF_VOICES_DIR", ref_dir)
    print(f"Demo input ready in {cv_dir}")
    print(f"Planted a wrong transcript on {planted} to show the transcript gate.")
    print("Data: Common Voice 17.0 (Mozilla Foundation, CC0 1.0) via fixie-ai/common_voice_17_0; "
          "LibriSpeech speaker 1272 (Panayotov, Chen, Povey, Khudanpur 2015, CC BY 4.0) via "
          "hf-internal-testing/librispeech_asr_dummy.")

if not SMOKE:
    if _env("COMMON_VOICE_DIR"):
        if _env("RAW_AUDIO_SOURCE") or _env("METADATA_CSV"):
            raise RuntimeError("Set COMMON_VOICE_DIR or RAW_AUDIO_SOURCE + METADATA_CSV, not both.")
        selected = os.path.join(WORK, "cv_selected")
        _derive("RAW_AUDIO_SOURCE", selected)
        _derive("METADATA_CSV", os.path.join(WORK, "cv_selected.csv"))
        select_common_voice(_env("COMMON_VOICE_DIR"), selected, _env("METADATA_CSV"))
    missing = [k for k in ("RAW_AUDIO_SOURCE", "METADATA_CSV", "REF_VOICES_DIR") if not _env(k)]
    if missing:
        raise RuntimeError(
            f"{', '.join(missing)} not set. Set COMMON_VOICE_DIR or RAW_AUDIO_SOURCE + METADATA_CSV, "
            f"and REF_VOICES_DIR; or leave every input empty to run the demo."
        )
    print(f"Inputs: audio {_env('RAW_AUDIO_SOURCE')} | transcripts {_env('METADATA_CSV')} | "
          f"voices {_env('REF_VOICES_DIR')} | language {_env('TTS_LANG')}")

---
### Step 1: Format Standardization
Converts mixed audio formats (mp3, ogg, flv, etc.) to standard WAV format.


In [ ]:
%pip install --quiet --pre "pydub>=0.25.1"

import os
import shutil
from pydub import AudioSegment

INPUT_DIR = _env("RAW_AUDIO_SOURCE")
OUTPUT_DIR = _env("DIR_CONVERTED")

SUPPORTED_FORMATS = {'.mp3', '.mp4', '.wma', '.ogg', '.flv', '.aac', '.m4a', '.amr', '.wav'}

def convert_dataset(input_path, output_path):
    os.makedirs(output_path, exist_ok=True)
    print(f"Converting from {input_path} -> {output_path}")

    for root, _, files in os.walk(input_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            base, ext = os.path.splitext(filename)

            if ext.lower() not in SUPPORTED_FORMATS:
                continue

            target_file = os.path.join(output_path, f"{base}.wav")

            if ext.lower() == '.wav':
                shutil.copy2(file_path, target_file)
            else:
                audio = AudioSegment.from_file(file_path)
                audio.export(target_file, format='wav')
                print(f"Converted: {filename}")

convert_dataset(INPUT_DIR, OUTPUT_DIR)


### Step 2: Audio Denoising

Removes background noise with [audiosronnx](https://github.com/TigreGotico/audiosronnx)'s
`load_denoise`, a pure-`onnxruntime` denoiser: no PyTorch or CUDA driver is needed at
runtime. `DENOISE_ENGINE` names the engine (`audiosronnx.available_denoisers()` prints the
rest); `DENOISE_<OPTION>` sets one constructor option of the selected engine, and setting an
option the engine does not accept stops the notebook, naming the engine and the option.


In [ ]:
%pip install --quiet --pre "audiosronnx>=0.9.0a2" "tqdm>=4.70.1"

onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()
if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

import inspect
import os

from audiosronnx import load_denoise
from audiosronnx.base import get_engine
from tqdm import tqdm

# option name -> typed-parsing kind; the env var is DENOISE_<OPTION UPPERCASE>
DENOISE_OPTION_TYPES = {
    "revision": "str",
    "onnx_path": "str",
    "mask_only": "bool",
    "enc_path": "str",
    "erb_dec_path": "str",
    "df_dec_path": "str",
    "model": "str",
    "attn_limit_db": "float",
    "dither": "float",
    "analysis_path": "str",
    "vocoder_path": "str",
    "mel_filters_path": "str",
}

INPUT_DIR = _env("DIR_CONVERTED")
OUTPUT_DIR = _env("DIR_DENOISED")
DENOISE_ENGINE = _env("DENOISE_ENGINE", "gtcrn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

engine_entry = get_engine(DENOISE_ENGINE)
adapter_params = inspect.signature(engine_entry.adapter_class.__init__).parameters
denoise_kwargs = {}
for option, kind in DENOISE_OPTION_TYPES.items():
    env_name = f"DENOISE_{option.upper()}"
    raw = _env(env_name)
    if not raw:
        continue
    if option not in adapter_params:
        raise RuntimeError(f"{env_name} is set but engine {DENOISE_ENGINE!r} has no {option!r} option")
    denoise_kwargs[option] = _parse_typed_env(env_name, raw, kind)
if "providers" in adapter_params:
    denoise_kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))

print(f"Loading audiosronnx denoiser {DENOISE_ENGINE!r}...")
denoiser = load_denoise(DENOISE_ENGINE, **denoise_kwargs)

files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".wav"))
for fname in tqdm(files, desc="Denoising Audio"):
    out_path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(out_path):
        continue
    denoiser.denoise_file(os.path.join(INPUT_DIR, fname), out_path)

print(f"Denoising: {len(files)} file(s) -> {OUTPUT_DIR}")


---
### Quality gate setup

Imports [speechonnxmetrics](https://github.com/TigreGotico/speechonnxmetrics) and defines the helpers the three gates share: the per-clip records, the gate log and the function that moves failing clips to `WORK_DIR/rejected/`.

In [ ]:
%pip install --quiet --pre "speechonnxmetrics[audio,speaker,asr]>=0.0.2a3" "soundfile>=0.14.0" "numpy>=2.0" "pandas>=2.2"

import csv
import json
import os
import shutil
from importlib.metadata import version as _pkg_version

import numpy as np
import pandas as pd
import soundfile as sf

from speechonnxmetrics import score_batch as som_score_batch

ENFORCE = _parse_typed_env("ENFORCE_GATES", _env("ENFORCE_GATES"), "bool")
PROVIDERS = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))
REJECT_DIR = os.path.join(_env("WORK_DIR"), "rejected")

# speechonnxmetrics result keys -> the score names of docs/manifest.md and tts_measure
NATURALNESS_KEYS = {
    "utmos": "utmos",
    "dnsmos.sig": "dnsmos_sig",
    "dnsmos.bak": "dnsmos_bak",
    "dnsmos.ovrl": "dnsmos_ovrl",
    "nisqa.mos": "nisqa_mos",
}

CLIPS = {}       # source clip file name -> record
CONVERTED = {}   # "<voice>/<file name>" -> record
GATE_LOG = []    # one entry per gate: counts, hours, threshold


def new_record(name, **fields):
    return {"clip": name, "scores": {}, "flags": [], "dropped_at": None, "reason": "", **fields}


def duration_s(path):
    info = sf.info(path)
    return info.frames / info.samplerate


def naturalness(paths, metrics):
    """One dict of manifest-named scores per path; {} when the instrument failed on it."""
    results = som_score_batch(paths, metrics, providers=PROVIDERS)
    out = []
    for res in results:
        out.append({
            name: float(res[key]) for key, name in NATURALNESS_KEYS.items() if res.get(key) is not None
        })
    return out


def close_gate(gate, registry, candidates, failures, threshold):
    """Record one gate. `candidates` are (key, path, duration_s); `failures` maps a key to
    the reason it failed. With ENFORCE_GATES the failing files move to rejected/<gate>/
    and every later step leaves them out."""
    for key, path, _dur in candidates:
        if key in failures:
            rec = registry[key]
            rec["flags"].append(gate)
            rec["reason"] = failures[key]
            if ENFORCE:
                rec["dropped_at"] = gate
                target = os.path.join(REJECT_DIR, gate, os.path.dirname(key))
                os.makedirs(target, exist_ok=True)
                shutil.move(path, os.path.join(target, os.path.basename(path)))
    kept = [c for c in candidates if not (ENFORCE and c[0] in failures)]
    entry = {
        "gate": gate, "threshold": threshold, "enforced": ENFORCE, "skipped": False,
        "n_in": len(candidates), "n_failed": len(failures), "n_out": len(kept),
        "hours_in": sum(c[2] for c in candidates) / 3600, "hours_out": sum(c[2] for c in kept) / 3600,
    }
    GATE_LOG.append(entry)
    verb = "dropped" if ENFORCE else "would drop (ENFORCE_GATES=false)"
    print(f"{gate}: {entry['n_in']} in, {verb} {entry['n_failed']}, {entry['n_out']} out | {threshold}")
    for key, reason in list(failures.items())[:5]:
        print(f"    {key}: {reason}")
    if len(failures) > 5:
        print(f"    ... and {len(failures) - 5} more (see quality_scores.csv)")
    if candidates and not kept:
        write_gate_report()
        raise RuntimeError(
            f"{gate} dropped every clip ({threshold}). The scores are in quality_scores.csv and the clips in "
            f"{REJECT_DIR}/{gate}/; lower the threshold (MIN_INPUT_DNSMOS_OVRL, MAX_CER, MIN_SPK_COS, MIN_UTMOS), "
            f"change the input, or set ENFORCE_GATES=false."
        )
    return [c[0] for c in kept]


def log_step(step, n_in, n_out, rule, dropped=()):
    """A pipeline step that is not a quality gate but drops clips, so the counts chain."""
    for name in dropped:
        rec = CLIPS.setdefault(name, new_record(name))
        rec["dropped_at"] = step
        rec["reason"] = rule
    GATE_LOG.append({
        "gate": step, "threshold": rule, "enforced": True, "skipped": False, "step": True,
        "n_in": n_in, "n_out": n_out, "n_failed": n_in - n_out, "hours_in": None, "hours_out": None,
    })
    print(f"{step}: {n_in} in, {n_out} out | {rule}")


def metadata_names(path):
    """File names (with .wav) listed in a `name|transcript` metadata file."""
    with open(path, newline="", encoding="utf-8") as f:
        names = set()
        for row in csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE):
            if len(row) >= 2:
                names.add(row[0] if row[0].endswith(".wav") else row[0] + ".wav")
    return names


def skip_gate(gate, n, reason):
    GATE_LOG.append({
        "gate": gate, "threshold": "", "enforced": ENFORCE, "skipped": True, "skip_reason": reason,
        "n_in": n, "n_failed": 0, "n_out": n, "hours_in": None, "hours_out": None,
    })
    print(f"{gate}: SKIPPED. {reason}")


SCORE_COLUMNS = [
    "dnsmos_sig_before", "dnsmos_bak_before", "dnsmos_ovrl_before", "nisqa_mos_before",
    "dnsmos_sig_after", "dnsmos_bak_after", "dnsmos_ovrl_after", "nisqa_mos_after",
    "hyp", "cer", "spk_cos_source", "spk_cos", "utmos",
]


def write_gate_report():
    """Write quality_scores.csv and gate_summary.json, print the summary and return it."""
    work_dir = _env("WORK_DIR")
    table_rows = []
    for kind, registry in (("source", CLIPS), ("converted", CONVERTED)):
        for key, rec in sorted(registry.items()):
            status = f"dropped at {rec['dropped_at']}" if rec["dropped_at"] else (
                f"kept, would fail {', '.join(rec['flags'])}" if rec["flags"] else "kept")
            table_rows.append({
                "clip": os.path.splitext(rec["clip"])[0], "stage": kind, "identity": rec.get("identity", ""),
                **{c: rec["scores"].get(c) for c in SCORE_COLUMNS},
                "status": status, "reason": rec["reason"],
            })
    table = pd.DataFrame(table_rows)
    table_path = os.path.join(work_dir, "quality_scores.csv")
    table.to_csv(table_path, index=False)

    final_dir = _env("DIR_FINAL_VC")
    kept_hours = {}
    if os.path.isdir(final_dir):
        for voice in sorted(os.listdir(final_dir)):
            voice_dir = os.path.join(final_dir, voice)
            if os.path.isdir(voice_dir):
                files = [f for f in os.listdir(voice_dir) if f.endswith(".wav") and f"{voice}/{f}" in CONVERTED]
                kept_hours[voice] = (len(files), sum(duration_s(os.path.join(voice_dir, f)) for f in files) / 3600)

    print(f"Quality gates and clip counts (enforced: {ENFORCE}; one clip per voice counts once at Gate 3)")
    for g in GATE_LOG:
        if g["skipped"]:
            print(f"  {g['gate']:22s} SKIPPED ({g['skip_reason'][:70]}...)")
            continue
        hours = f"  hours out {g['hours_out']:.4f}" if g["hours_out"] is not None else ""
        print(f"  {g['gate']:22s} in {g['n_in']:4d}  out {g['n_out']:4d}  "
              f"{'dropped' if (ENFORCE or g.get('step')) else 'would drop'} {g['n_failed']:4d}{hours}")
        print(f"  {'':22s} rule: {g['threshold']}")
    for voice, (n, hours) in kept_hours.items():
        print(f"Kept for voice {voice}: {n} clip(s), {hours:.4f} h")

    print()
    print("Score distributions (n, p10 / median / p90)")
    distributions = {}
    for column in SCORE_COLUMNS:
        if column == "hyp":
            continue
        by_stage = {stage: pd.to_numeric(table.loc[table["stage"] == stage, column], errors="coerce").dropna().to_numpy()
                    for stage in ("source", "converted")} if len(table) else {}
        present = {stage: v for stage, v in by_stage.items() if len(v)}
        for stage, values in present.items():
            name = column if len(present) == 1 else f"{column}_{stage}"
            p10, p50, p90 = np.percentile(values, [10, 50, 90])
            distributions[name] = {"n": int(len(values)), "p10": float(p10), "p50": float(p50), "p90": float(p90)}
            print(f"  {name:22s} n={len(values):4d}  {p10:7.3f} / {p50:7.3f} / {p90:7.3f}")

    summary = {
        "enforced": ENFORCE, "gates": GATE_LOG, "distributions": distributions,
        "kept": {voice: {"clips": n, "hours": h} for voice, (n, h) in kept_hours.items()},
    }
    with open(os.path.join(work_dir, "gate_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print()
    print(f"Per-clip table: {table_path}")
    print(f"Gate summary  : {os.path.join(work_dir, 'gate_summary.json')}")
    return summary

---
### Gate 1: Input quality after denoising

Scores every clip with [speechonnxmetrics](https://github.com/TigreGotico/speechonnxmetrics) before denoising (`DIR_CONVERTED`) and after it (`DIR_DENOISED`), prints the denoiser's median effect, and drops clips whose denoised score is under the minimum. `QUALITY_INPUT_METRICS` selects `dnsmos` (default, MIT weights), `nisqa` (CC BY-NC-SA weights, non-commercial use only) or both. Both scales run from 1 to 5; the default minimum of 2.5 (`MIN_INPUT_DNSMOS_OVRL`, `MIN_INPUT_NISQA_MOS`) is the middle of the poor-to-fair band. A clip that cannot be scored fails the gate. With `ENFORCE_GATES=true` (the default outside SMOKE) failing clips move to `WORK_DIR/rejected/gate1_input_quality/`; with `false` they stay and the table only marks them.

In [ ]:
input_metrics = [m.strip() for m in _env("QUALITY_INPUT_METRICS").split(",") if m.strip()]
unknown = [m for m in input_metrics if m not in ("dnsmos", "nisqa")]
if unknown or not input_metrics:
    raise RuntimeError(f"QUALITY_INPUT_METRICS={_env('QUALITY_INPUT_METRICS')!r}: use dnsmos and/or nisqa")
# metric name in the table -> minimum, for the families that were scored
input_minimums = {}
if "dnsmos" in input_metrics:
    input_minimums["dnsmos_ovrl"] = float(_env("MIN_INPUT_DNSMOS_OVRL"))
if "nisqa" in input_metrics:
    input_minimums["nisqa_mos"] = float(_env("MIN_INPUT_NISQA_MOS"))

# Only this run's clips: a workspace reused after another CV_SPEAKER or input holds older ones.
current_names = metadata_names(_env("METADATA_CSV"))
denoised_files = sorted(f for f in os.listdir(_env("DIR_DENOISED")) if f.endswith(".wav") and f in current_names)
stale = len([f for f in os.listdir(_env("DIR_DENOISED")) if f.endswith(".wav")]) - len(denoised_files)
if stale:
    print(f"Ignoring {stale} clip(s) in {_env('DIR_DENOISED')} left by an earlier run with other inputs.")
before = naturalness([os.path.join(_env("DIR_CONVERTED"), f) for f in denoised_files], input_metrics)
after = naturalness([os.path.join(_env("DIR_DENOISED"), f) for f in denoised_files], input_metrics)

candidates, failures = [], {}
for fname, b, a in zip(denoised_files, before, after):
    path = os.path.join(_env("DIR_DENOISED"), fname)
    rec = CLIPS.setdefault(fname, new_record(fname))
    rec["scores"].update({f"{k}_before": v for k, v in b.items()})
    rec["scores"].update({f"{k}_after": v for k, v in a.items()})
    candidates.append((fname, path, duration_s(path)))
    for metric, minimum in input_minimums.items():
        value = a.get(metric)
        if value is None:
            failures[fname] = f"{metric} could not be scored"
        elif value < minimum:
            failures[fname] = f"{metric} {value:.2f} < {minimum}"

for metric in input_minimums:
    deltas = [r["scores"][f"{metric}_after"] - r["scores"][f"{metric}_before"] for r in CLIPS.values()
              if f"{metric}_after" in r["scores"] and f"{metric}_before" in r["scores"]]
    if deltas:
        print(f"Denoiser effect on {metric}: median {np.median(deltas):+.2f} "
              f"(before {np.median([r['scores'][f'{metric}_before'] for r in CLIPS.values() if f'{metric}_before' in r['scores']]):.2f}, "
              f"after {np.median([r['scores'][f'{metric}_after'] for r in CLIPS.values() if f'{metric}_after' in r['scores']]):.2f}); "
              f"improved on {sum(d > 0 for d in deltas)} of {len(deltas)} clip(s)")

KEPT1 = set(close_gate(
    "gate1_input_quality", CLIPS, candidates, failures,
    ", ".join(f"{m} >= {v}" for m, v in input_minimums.items()),
))

if SMOKE:
    # The smoke fixtures carry one clip that must fail: if Gate 1 stops dropping, CI goes red.
    assert ENFORCE, "the smoke path must run the enforcing branch"
    assert set(failures) == {"sample_bad.wav"}, f"Gate 1 should fail exactly the planted clip, failed {sorted(failures)}"
    assert os.path.isfile(os.path.join(REJECT_DIR, "gate1_input_quality", "sample_bad.wav")), "rejected clip was not moved"
    assert not os.path.exists(os.path.join(_env("DIR_DENOISED"), "sample_bad.wav")), "rejected clip is still in the pipeline"

### Step 3: Silence Trimming
Trims leading/trailing silence using `librosa`. Removes files that are too short after trimming.


In [ ]:
%pip install --quiet --pre "librosa>=0.10.2" "soundfile>=0.14.0"

import os
import librosa
import soundfile as sf
from glob import glob
from tqdm import tqdm

INPUT_DIR = _env("DIR_DENOISED")
OUTPUT_DIR = _env("DIR_TRIMMED")
MIN_DURATION = float(_env("MIN_TRIM_DURATION", "1.0"))

def trim_silence(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = glob(os.path.join(input_dir, '*.wav'))

    for fpath in tqdm(files, desc="Trimming Silence"):
        fname = os.path.basename(fpath)
        out_path = os.path.join(output_dir, fname)

        audio, sr = librosa.load(fpath, sr=None, mono=True)
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=40)
        duration = len(trimmed_audio) / sr

        if duration >= MIN_DURATION:
            sf.write(out_path, trimmed_audio, sr)

trim_silence(INPUT_DIR, OUTPUT_DIR)

trimmed_ok = KEPT1 & set(os.listdir(OUTPUT_DIR))
log_step("step3_trim", len(KEPT1), len(trimmed_ok), f"duration >= {MIN_DURATION} s after trimming",
         dropped=KEPT1 - trimmed_ok)


### Step 4: Volume Normalization
Normalizes audio to a target loudness (e.g., -20.0 LUFS/dBFS) to ensure consistency.


In [ ]:
%pip install --quiet --pre "pydub>=0.25.1"

import os
from pydub import AudioSegment
from glob import glob
from tqdm import tqdm

INPUT_DIR = _env("DIR_TRIMMED")
OUTPUT_DIR = _env("DIR_NORMALIZED")
TARGET_DBFS = float(_env("TARGET_LUFS", "-20.0"))

def normalize_volume(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = glob(os.path.join(input_dir, '*.wav'))

    for fpath in tqdm(files, desc="Normalizing Volume"):
        fname = os.path.basename(fpath)
        out_path = os.path.join(output_dir, fname)

        audio = AudioSegment.from_wav(fpath)
        gain = TARGET_DBFS - audio.dBFS
        normalized = audio.apply_gain(gain)
        normalized.export(out_path, format="wav")

normalize_volume(INPUT_DIR, OUTPUT_DIR)


### Step 5: WPM (Words Per Minute) Filtering
Filters out outliers based on speaking rate using the provided transcript CSV.


In [ ]:
%pip install --quiet --pre "pandas>=2.2" "soundfile>=0.14.0"

import csv
import os
import re
import shutil
import pandas as pd
import soundfile as sf

INPUT_DIR = _env("DIR_NORMALIZED")
CSV_PATH = _env("METADATA_CSV")
OUTPUT_DIR = _env("DIR_FILTERED")
STD_THRESHOLD = float(_env("WPM_STD_THRESHOLD", "2.0"))

def calculate_wpm(row, audio_dir):
    file_path = os.path.join(audio_dir, str(row['file_name']))
    if not file_path.endswith('.wav') and not os.path.exists(file_path):
        file_path += ".wav"

    cleaned_transcript = re.sub(r'[^\w\s]', '', str(row['transcript'])).strip().lower()
    word_count = len(cleaned_transcript.split())

    if word_count == 0 or not os.path.exists(file_path):
        return 0.0

    try:
        with sf.SoundFile(file_path) as f:
            duration_min = (len(f) / f.samplerate) / 60.0
        return word_count / duration_min if duration_min > 0 else 0.0
    except Exception:
        return 0.0

def filter_wpm(input_dir, csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(csv_path, sep='|', names=['file_name', 'transcript'], header=None, quoting=csv.QUOTE_NONE)

    print("Calculating WPM...")
    df['WPM'] = df.apply(lambda row: calculate_wpm(row, input_dir), axis=1)
    valid_data = df[df['WPM'] > 0]

    mean_wpm = valid_data['WPM'].mean()
    std_wpm = valid_data['WPM'].std()
    lower = mean_wpm - (STD_THRESHOLD * std_wpm)
    upper = mean_wpm + (STD_THRESHOLD * std_wpm)

    print(f"Mean WPM: {mean_wpm:.2f} | Range: {lower:.2f} - {upper:.2f}")

    filtered = valid_data[(valid_data['WPM'] >= lower) & (valid_data['WPM'] <= upper)]

    print(f"Copying {len(filtered)} valid files...")
    new_meta = []
    for _, row in filtered.iterrows():
        fname = str(row['file_name'])
        if not fname.endswith('.wav'):
            fname += ".wav"

        src = os.path.join(input_dir, fname)
        dst = os.path.join(output_dir, fname)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            new_meta.append(f"{fname}|{row['transcript']}")

    with open(os.path.join(output_dir, "metadata_filtered.csv"), "w") as f:
        f.write("\n".join(new_meta))

filter_wpm(INPUT_DIR, CSV_PATH, OUTPUT_DIR)

wpm_kept = {n for n in metadata_names(os.path.join(OUTPUT_DIR, "metadata_filtered.csv"))
            if os.path.isfile(os.path.join(OUTPUT_DIR, n))} & KEPT1
log_step("step5_wpm", len(KEPT1 & set(os.listdir(INPUT_DIR))), len(wpm_kept),
         f"words per minute within {STD_THRESHOLD} standard deviations of the mean",
         dropped=(KEPT1 & set(os.listdir(INPUT_DIR))) - wpm_kept)


---
### Gate 2: Transcript check

Transcribes each cleaned clip and compares the result with its given transcript. Character error rate above `MAX_CER` (default 0.25) drops the clip; this catches wrong or mismatched Common Voice sentences. The recogniser is an [onnx-asr](https://github.com/istupakov/onnx-asr) model: `ASR_MODEL` names it, and the default is `nemo-parakeet-tdt-0.6b-v3`, which covers 25 European languages (`ASR_QUANTIZATION=int8` keeps it near 670 MB). `ASR_STT_MODULE` names an OVOS STT plugin instead. Whisper models are refused. When no model covers `TTS_LANG`, the gate is skipped and prints why, and the table has no `cer` column values.

In [ ]:
import shlex
import unicodedata

from speechonnxmetrics.asr import (
    EmptyReferenceError, Normalizer, cer as som_cer, collapse_whitespace, lowercase, strip_punctuation,
)

GATE2 = "gate2_transcript"
# Languages the default model (NVIDIA Parakeet TDT 0.6B v3, run by onnx-asr) transcribes.
DEFAULT_ASR_MODEL = "nemo-parakeet-tdt-0.6b-v3"
DEFAULT_ASR_LANGS = {
    "bg", "hr", "cs", "da", "nl", "en", "et", "fi", "fr", "de", "el", "hu", "it", "lv", "lt",
    "mt", "pl", "pt", "ro", "ru", "sk", "sl", "es", "sv", "uk",
}
# Same normalisation as tts_measure's cer: NFC, casefold, punctuation out, spaces collapsed.
CER_NORMALIZER = Normalizer((
    lambda text: unicodedata.normalize("NFC", text), lowercase, strip_punctuation, collapse_whitespace,
))

filtered_dir = _env("DIR_FILTERED")
meta_path = os.path.join(filtered_dir, "metadata_filtered.csv")
with open(meta_path, newline="", encoding="utf-8") as f:
    transcripts = {}
    for row in csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE):
        if len(row) >= 2:
            transcripts[row[0] if row[0].endswith(".wav") else row[0] + ".wav"] = row[1]

filtered_files = sorted(f for f in os.listdir(filtered_dir) if f.endswith(".wav") and f in transcripts and f in KEPT1)
lang = _env("TTS_LANG").split("-")[0].lower()
asr_model, stt_module = _env("ASR_MODEL").strip(), _env("ASR_STT_MODULE").strip()


import re

# Whisper checkpoints are often named without the word: faster-whisper and distil-whisper accept
# `large-v3`, `distil-large-v3`, `large-v3-turbo` and `turbo`, and their ctranslate2 exports have
# their own repository names. A configuration or a loaded plugin that mentions any of these is refused.
WHISPER_PATTERN = re.compile(
    r"whisper|distil|ctranslate2|faster[-_ ]?whisper|\bturbo\b|[-_/\"' :]large[-_]?v[123]\b",
    re.IGNORECASE,
)


def refuse_whisper(*named):
    for name in named:
        if WHISPER_PATTERN.search(str(name)):
            raise RuntimeError(f"{str(name)[:80]!r} looks like a Whisper model: Whisper models are not used for the "
                               f"transcript check; pick another onnx-asr model or an OVOS STT plugin.")


if SMOKE:
    # The refusal is code CI can run: it must catch Whisper checkpoints that do not say "whisper".
    for hidden in ("distil-large-v3", "large-v3-turbo", "turbo", "Systran/faster-distil-large-v3",
                   '{"model": "openai/whisper-large-v3"}'):
        try:
            refuse_whisper(hidden)
        except RuntimeError:
            continue
        raise AssertionError(f"refuse_whisper let {hidden!r} through")
    refuse_whisper("nemo-parakeet-tdt-0.6b-v3", "{}", "ovos-stt-plugin-vosk")


# The names, and the whole plugin configuration: a generic plugin can be pointed at a Whisper
# checkpoint through its config.
refuse_whisper(asr_model, stt_module, _env("ASR_STT_CONFIG"))
recognise = None  # Gate 3 scores the converted clips with the same recogniser, when there is one
max_cer = float(_env("MAX_CER"))

if SMOKE:
    skip_reason = "SMOKE=True: the fixtures are sine tones with no speech, and the default model is a 670 MB download."
elif stt_module or asr_model or lang in DEFAULT_ASR_LANGS:
    skip_reason = None
else:
    skip_reason = (f"no ASR model covers language {lang!r}: the default ({DEFAULT_ASR_MODEL}) covers "
                   f"{', '.join(sorted(DEFAULT_ASR_LANGS))}. Set ASR_MODEL (an onnx-asr model name) or "
                   f"ASR_STT_MODULE (an OVOS STT plugin) to check transcripts for {lang!r}.")

if skip_reason:
    skip_gate(GATE2, len(filtered_files), skip_reason)
    asr_instrument = None
    KEPT2 = set(filtered_files)
else:
    if stt_module:
        %pip install --quiet --pre "ovos-plugin-manager>=2.12.4a1"
        %pip install --quiet --pre {shlex.quote(stt_module)}
        from ovos_plugin_manager.stt import load_stt_plugin
        from ovos_plugin_manager.utils.audio import AudioData

        stt_config = json.loads(_env("ASR_STT_CONFIG", "{}") or "{}")
        stt_config.setdefault("lang", _env("TTS_LANG"))
        stt = load_stt_plugin(stt_module)(config=stt_config)
        refuse_whisper(type(stt).__module__, type(stt).__name__, [c.__module__ for c in type(stt).__mro__],
                       getattr(stt, "config", ""), getattr(stt, "model", ""))
        print(f"Transcript gate: OVOS STT plugin {stt_module}")

        def recognise(path):
            return stt.execute(AudioData.from_file(path), language=_env("TTS_LANG"))

        asr_instrument = {"package": stt_module, "version": _pkg_version(stt_module)}
    else:
        import onnx_asr

        model_name = asr_model or DEFAULT_ASR_MODEL
        print(f"Transcript gate: onnx-asr model {model_name} ({_env('ASR_QUANTIZATION') or 'unquantized'}); "
              f"the first run downloads it")
        asr = onnx_asr.load_model(model_name, quantization=_env("ASR_QUANTIZATION") or None, providers=PROVIDERS)

        def recognise(path):
            return asr.recognize(path)

        asr_instrument = {"package": "onnx-asr", "version": _pkg_version("onnx-asr"), "model": model_name}

    candidates, failures = [], {}
    for fname in tqdm(filtered_files, desc="ASR round trip"):
        path = os.path.join(filtered_dir, fname)
        rec = CLIPS.setdefault(fname, new_record(fname))
        candidates.append((fname, path, duration_s(path)))
        text = transcripts.get(fname, "")
        try:
            hyp = recognise(path)
        except Exception as exc:
            failures[fname] = f"ASR failed: {exc}"
            continue
        rec["scores"]["hyp"] = hyp
        try:
            rec["scores"]["cer"] = float(som_cer(text, hyp, normalizer=CER_NORMALIZER))
        except EmptyReferenceError:
            failures[fname] = "empty transcript"
            continue
        if rec["scores"]["cer"] > max_cer:
            failures[fname] = f"cer {rec['scores']['cer']:.2f} > {max_cer}; heard {hyp!r}, transcript {text!r}"

    close_gate(GATE2, CLIPS, candidates, failures, f"cer <= {max_cer}")
    KEPT2 = set(c[0] for c in candidates) - (set(failures) if ENFORCE else set())
    if ENFORCE and failures:
        with open(meta_path, "w", encoding="utf-8") as f:
            f.write("\n".join(f"{name}|{transcripts[name]}" for name in filtered_files
                              if name in transcripts and name not in failures))

### Step 6: Voice Conversion

Converts every filtered clip to one or more target voices with
[voiceclonnx](https://github.com/TigreGotico/voiceclonnx), a pure-`onnxruntime`
voice-conversion library: no PyTorch or CUDA driver is needed at runtime. Every WAV
in `REF_VOICES_DIR` names one target voice; each filtered clip is converted to every
reference voice and written under its own subdirectory of `DIR_FINAL_VC`.

`VC_ENGINE` names the engine (`voiceclonnx list` prints the rest); `VC_<OPTION>` sets
one constructor option of the selected engine (`VC_QUANTIZED`, `VC_EXAGGERATION`,
`VC_ODE_STEPS`, `VC_CHUNK_SAMPLES`, `VC_K`, `VC_DEFAULT_MODEL`, `VC_SAMPLE_RATE`,
`VC_SPEAKER_ID`, `VC_F0_UP_KEY`), and setting one the engine does not accept stops
the notebook, naming the engine and the option.


In [ ]:
%pip install --quiet --pre "voiceclonnx>=0.0.3a3"

onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()
if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

import inspect
import os
import shutil

from voiceclonnx import VoiceCloner
from voiceclonnx.engines.base import get_engine

# option name -> typed-parsing kind; the env var is VC_<OPTION UPPERCASE>
VC_OPTION_TYPES = {
    "quantized": "bool",
    "exaggeration": "float",
    "ode_steps": "int",
    "chunk_samples": "int",
    "k": "int",
    "default_model": "str",
    "sample_rate": "int",
    "speaker_id": "int",
    "f0_up_key": "int",
}

INPUT_DIR = _env("DIR_FILTERED")
OUTPUT_BASE = _env("DIR_FINAL_VC")
REFS_FOLDER = _env("REF_VOICES_DIR")
VC_ENGINE = _env("VC_ENGINE", "chatterbox")

ref_voices = sorted(f for f in os.listdir(REFS_FOLDER) if f.endswith(".wav"))
if not ref_voices:
    raise RuntimeError(f"REF_VOICES_DIR={REFS_FOLDER} has no .wav reference voices")

source_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".wav") and f in KEPT2)
if not source_files:
    raise RuntimeError(f"{INPUT_DIR} has no filtered .wav clips to convert")

if SMOKE:
    # bound the smoke path regardless of how many voices REF_VOICES_DIR holds,
    # and keep its download small even if VC_QUANTIZED was overridden.
    ref_voices = ref_voices[:2]
    os.environ["VC_QUANTIZED"] = "true"

engine_entry = get_engine(VC_ENGINE)
adapter_params = inspect.signature(engine_entry.adapter_class.__init__).parameters
vc_kwargs = {}
for option, kind in VC_OPTION_TYPES.items():
    env_name = f"VC_{option.upper()}"
    raw = _env(env_name)
    if not raw:
        continue
    if option not in adapter_params:
        raise RuntimeError(f"{env_name} is set but engine {VC_ENGINE!r} has no {option!r} option")
    vc_kwargs[option] = _parse_typed_env(env_name, raw, kind)
if "providers" in adapter_params:
    vc_kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))

print(f"Loading voiceclonnx engine {VC_ENGINE!r}...")
cloner = VoiceCloner(engine=VC_ENGINE, **vc_kwargs)

# Converted clips left in a voice directory by an earlier run with other inputs must not reach
# the dataset a trainer reads.
stale_dir = os.path.join(_env("WORK_DIR"), "rejected", "stale")
for ref_voice in ref_voices:
    voice_id = os.path.splitext(ref_voice)[0]
    voice_out_dir = os.path.join(OUTPUT_BASE, voice_id)
    if os.path.isdir(voice_out_dir):
        for old in sorted(os.listdir(voice_out_dir)):
            if old.endswith(".wav") and old not in source_files:
                os.makedirs(os.path.join(stale_dir, voice_id), exist_ok=True)
                shutil.move(os.path.join(voice_out_dir, old), os.path.join(stale_dir, voice_id, old))
                print(f"Moved stale converted clip {voice_id}/{old} to rejected/stale")

converted = 0
for ref_voice in ref_voices:
    voice_id = os.path.splitext(ref_voice)[0]
    ref_path = os.path.join(REFS_FOLDER, ref_voice)
    voice_out_dir = os.path.join(OUTPUT_BASE, voice_id)
    os.makedirs(voice_out_dir, exist_ok=True)

    for fname in source_files:
        out_file = os.path.join(voice_out_dir, fname)
        if os.path.exists(out_file):
            continue
        cloner.clone_voice(os.path.join(INPUT_DIR, fname), ref_path, out_file)
        converted += 1

print(f"Voice conversion: {converted} file(s) written across {len(ref_voices)} voice(s) -> {OUTPUT_BASE}")


---
### Gate 3: Converted voice

Checks each converted clip against the target voice. `spk_cos` is the cosine between the speaker embedding of the clip and of its reference WAV (speakeronnx, WeSpeaker ResNet34, the same instrument as `tts_measure.ipynb`); different speakers fall near 0 to 0.2 and one speaker scores 0.5 or more, so the default minimum `MIN_SPK_COS=0.3` rejects conversions that lost the target voice. `utmos` is the naturalness MOS on a 1 to 5 scale, where natural speech scores above about 3.5; the default minimum `MIN_UTMOS=3.0` is the "fair" point (`SCORE_UTMOS=false` skips it and its 411 MB download). The converted clip also goes through the transcript check of Gate 2 (`cer <= MAX_CER`, when that gate ran), because the converted audio is what a TTS model trains on; `hyp` and `cer` are written on the converted rows. Failing clips move to `WORK_DIR/rejected/gate3_converted/<voice>/`, so the manifest and every voice's `metadata.csv` list only clips that passed. `quality_scores.csv` and `gate_summary.json` are written right after this gate by the summary cell (or by the gate itself when it drops every clip). At this gate the counts are converted clips, one per source clip and voice. The table also keeps `spk_cos_source`, the same cosine for the unconverted clip, to show how far conversion moved the voice. The default engine is `chatterbox` (voiceclonnx's audio-to-audio cloner with natural prosody and a strong source-to-target shift); with a weaker engine most clips can fail this gate, in which case lower `MIN_SPK_COS` or change `VC_ENGINE` rather than trusting the survivors.

Chatterbox conversion is stochastic: the same clip converts differently on each run, so the kept count varies between runs of an unchanged notebook (a rerun converts again only the clips that are missing, so clips that failed are drawn again). Compare runs by the score distributions, not by the kept count.


In [ ]:
from speakeronnx import SpeakerEmbedder, cosine

GATE3 = "gate3_converted"
min_spk, min_utmos = float(_env("MIN_SPK_COS")), float(_env("MIN_UTMOS"))
score_utmos = _parse_typed_env("SCORE_UTMOS", _env("SCORE_UTMOS"), "bool")

print("Loading speaker embedder (wespeaker-resnet34)...")
embedder = SpeakerEmbedder(model="wespeaker-resnet34", providers=PROVIDERS)

# Pass 1: score every converted clip.
scored = []  # (key, voice, fname, path, quality)
candidates = []
for voice in sorted(os.listdir(_env("DIR_FINAL_VC"))):
    voice_dir = os.path.join(_env("DIR_FINAL_VC"), voice)
    if not os.path.isdir(voice_dir):
        continue
    ref_embedding = embedder.embed(os.path.join(_env("REF_VOICES_DIR"), voice + ".wav"))
    names = sorted(f for f in os.listdir(voice_dir) if f.endswith(".wav") and f in KEPT2)
    paths = [os.path.join(voice_dir, f) for f in names]
    utmos = naturalness(paths, ["utmos"]) if score_utmos else [{} for _ in paths]
    for fname, path, quality in zip(names, paths, utmos):
        key = f"{voice}/{fname}"
        rec = CONVERTED.setdefault(key, new_record(fname, identity=voice))
        rec["scores"]["spk_cos"] = float(cosine(embedder.embed(path), ref_embedding))
        rec["scores"].update(quality)
        # The same instrument on the unconverted clip, so the table shows how far
        # conversion moved the voice toward the reference.
        source_path = os.path.join(_env("DIR_FILTERED"), fname)
        if os.path.isfile(source_path):
            rec["scores"]["spk_cos_source"] = float(cosine(embedder.embed(source_path), ref_embedding))
        if recognise is not None:
            # The converted clip is what a TTS model trains on, so it gets the transcript check too.
            try:
                hyp = recognise(path)
                rec["scores"]["hyp"] = hyp
                rec["scores"]["cer"] = float(som_cer(transcripts[fname], hyp, normalizer=CER_NORMALIZER))
            except Exception as exc:
                rec["asr_error"] = str(exc)
        candidates.append((key, path, duration_s(path)))
        scored.append((key, voice, fname, path, quality))

# Similarity threshold per voice. Normally MIN_SPK_COS. The smoke path has tone fixtures with no
# meaningful similarity scale, so it puts the threshold between the two lowest scores of each
# voice, which makes exactly the lowest clip of each voice the one that must fail.
min_spk_by_voice = {}
if SMOKE:
    for voice in sorted({s[1] for s in scored}):
        ranked = sorted(CONVERTED[s[0]]["scores"]["spk_cos"] for s in scored if s[1] == voice)
        assert len(ranked) >= 2, f"the smoke path needs at least two clips for voice {voice}"
        min_spk_by_voice[voice] = (ranked[0] + ranked[1]) / 2

# Pass 2: apply the rules.
failures = {}
for key, voice, fname, path, quality in scored:
    rec = CONVERTED[key]
    limit = min_spk_by_voice.get(voice, min_spk)
    reasons = []
    if rec["scores"]["spk_cos"] < limit:
        reasons.append(f"spk_cos {rec['scores']['spk_cos']:.2f} < {limit:.2f}")
    if score_utmos and "utmos" not in quality:
        reasons.append("utmos could not be scored")
    elif score_utmos and quality["utmos"] < min_utmos:
        reasons.append(f"utmos {quality['utmos']:.2f} < {min_utmos}")
    if recognise is not None:
        if "asr_error" in rec:
            reasons.append(f"ASR on the converted clip failed: {rec['asr_error']}")
        elif rec["scores"]["cer"] > max_cer:
            reasons.append(f"cer {rec['scores']['cer']:.2f} > {max_cer}")
    if reasons:
        failures[key] = "; ".join(reasons)

if SMOKE:
    expected = {min(((CONVERTED[s[0]]["scores"]["spk_cos"], s[0]) for s in scored if s[1] == v))[1]
                for v in min_spk_by_voice}
    assert set(failures) == expected, \
        f"Gate 3 must drop exactly the lowest-scoring clip of each voice: expected {sorted(expected)}, got {sorted(failures)}"

threshold = (f"spk_cos >= {min_spk}" + (f", utmos >= {min_utmos}" if score_utmos else "")
             + (f", cer <= {max_cer}" if recognise is not None else ""))
if SMOKE:
    threshold = "spk_cos >= the midpoint between each voice's two lowest scores (smoke)"
if not score_utmos:
    print("UTMOS not scored (SCORE_UTMOS=false); the gate uses speaker similarity only.")
close_gate(GATE3, CONVERTED, candidates, failures, threshold)
if SMOKE:
    assert all(not os.path.exists(os.path.join(_env("DIR_FINAL_VC"), *k.split("/"))) for k in failures), \
        "Gate 3 left a failing clip in the voice directory"
    assert all(os.path.isfile(os.path.join(REJECT_DIR, GATE3, *k.split("/"))) for k in failures), \
        "Gate 3 did not move a failing clip to rejected/"

---
### Gate summary

`quality_scores.csv` (one row per source clip and per converted clip, with the before and after denoising scores, `hyp`, `cer`, `spk_cos_source`, `spk_cos`, `utmos` and why a clip was dropped) and `gate_summary.json` are written here, which also prints how many clips and hours each gate and each dropping step (`step3_trim`, `step5_wpm`) kept, and the p10, median and p90 of each score. The counts chain: each entry's `in` is the previous entry's `out`. This cell prints the summary again and stops with an error naming the gate when no converted clip survived.

In [ ]:
report = write_gate_report()
if not report["kept"] or not any(v["clips"] for v in report["kept"].values()):
    raise RuntimeError("No converted clip survived gate3_converted; see quality_scores.csv and rejected/gate3_converted/ "
                       "and lower MIN_SPK_COS, MIN_UTMOS or MAX_CER, or change VC_ENGINE.")

---
### Step 7: Punctuation Restoration

Restores punctuation on the transcript that every `DIR_FINAL_VC` voice shares
(`DIR_FILTERED`'s `metadata_filtered.csv`; voice conversion changes only the audio,
never the text). No in-house package restores punctuation, so this step keeps a
third-party checkpoint, [oliverguhr/fullstop-punctuation-multilang-large](https://huggingface.co/oliverguhr/fullstop-punctuation-multilang-large),
but runs its `onnx` export through `onnxruntime` alone: no PyTorch, TensorFlow or
`transformers` at runtime, matching the rest of this notebook. `PUNCT_MODEL` names
the Hugging Face repository and `PUNCT_MODEL_SUBDIR` the directory inside it that
holds the onnx export (`config.json`, `tokenizer.json`, `model.onnx`). The result
is written to `metadata_punct.csv` under `WORK_DIR`.

Long or unpunctuated transcripts benefit most, since a later training step may
split on sentence boundaries.

Off by default for the smoke path, the demo and Common Voice input (`RESTORE_PUNCTUATION=false`): the default checkpoint downloads over 1 GB of weights, and Common Voice sentences are already punctuated. Set `RESTORE_PUNCTUATION=true` for a corpus whose transcripts have none.


In [ ]:
import os

if not _parse_typed_env("RESTORE_PUNCTUATION", _env("RESTORE_PUNCTUATION"), "bool"):
    print(f"Punctuation restoration skipped (RESTORE_PUNCTUATION=false, the default for the smoke path, the demo "
          f"and Common Voice, whose sentences are punctuated): {_env('PUNCT_MODEL')} downloads over 1 GB of weights.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0" "tokenizers>=0.20,<1.0.0" "huggingface_hub>=0.36.0" "numpy>=2.0"

    import csv
    import json
    import re

    import numpy as np
    import onnxruntime as ort
    import pandas as pd
    from huggingface_hub import snapshot_download
    from tokenizers import Tokenizer

    CSV_PATH = os.path.join(_env("DIR_FILTERED"), "metadata_filtered.csv")
    OUTPUT_PATH = os.path.join(_env("WORK_DIR"), "metadata_punct.csv")
    PUNCT_MODEL = _env("PUNCT_MODEL", "oliverguhr/fullstop-punctuation-multilang-large")
    PUNCT_MODEL_SUBDIR = _env("PUNCT_MODEL_SUBDIR", "onnx")

    class OnnxPunctuationModel:
        """Restores punctuation with an onnxruntime session, following the chunk
        and merge algorithm of deepmultilingualpunctuation.PunctuationModel
        (github.com/oliverguhr/deepmultilingualpunctuation, MIT licence), with no
        PyTorch or transformers pipeline at runtime: a fast tokenizer plus the
        checkpoint's own onnx export do the same job."""

        def __init__(self, repo_id, subdir):
            local_dir = snapshot_download(
                repo_id=repo_id,
                allow_patterns=[f"{subdir}/*"],
                local_dir=os.path.join(_env("WORK_DIR"), "punct_model"),
            )
            model_dir = os.path.join(local_dir, subdir)
            providers = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))
            self.session = ort.InferenceSession(os.path.join(model_dir, "model.onnx"), providers=providers)
            self.tokenizer = Tokenizer.from_file(os.path.join(model_dir, "tokenizer.json"))
            self.tokenizer.enable_truncation(max_length=self._max_length(model_dir))
            with open(os.path.join(model_dir, "config.json"), encoding="utf-8") as f:
                self.id2label = json.load(f)["id2label"]

        @staticmethod
        def _max_length(model_dir):
            """Token budget per chunk: the checkpoint's own tokenizer_config.json
            model_max_length, so a chunk that would run past it is truncated the
            same way transformers.pipeline truncates by default, and caught below
            rather than fed past the model's position-embedding table."""
            config_path = os.path.join(model_dir, "tokenizer_config.json")
            if os.path.isfile(config_path):
                with open(config_path, encoding="utf-8") as f:
                    raw = json.load(f).get("model_max_length")
                if isinstance(raw, (int, float)) and 0 < raw <= 100_000:
                    return int(raw)
            return 512

        def _tag(self, text):
            enc = self.tokenizer.encode(text)
            input_ids = np.array([enc.ids], dtype=np.int64)
            attention_mask = np.array([enc.attention_mask], dtype=np.int64)
            logits = self.session.run(
                ["logits"], {"input_ids": input_ids, "attention_mask": attention_mask}
            )[0][0]
            exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
            probs = exp / exp.sum(axis=-1, keepdims=True)
            labels = probs.argmax(axis=-1)
            tagged = []
            for i in range(len(enc.ids)):
                if enc.special_tokens_mask[i]:
                    continue
                tagged.append({"entity": self.id2label[str(int(labels[i]))], "end": enc.offsets[i][1]})
            return tagged

        @staticmethod
        def _preprocess(text):
            return re.sub(r"(?<!\d)[.,;:!?](?!\d)", "", text).split()

        @staticmethod
        def _overlap_chunks(words, n, stride):
            for i in range(0, len(words), n - stride):
                yield words[i:i + n]

        def restore_punctuation(self, text, chunk_size=230):
            words = self._preprocess(text)
            if not words:
                return text
            overlap = 0 if len(words) <= chunk_size else 5
            batches = list(self._overlap_chunks(words, chunk_size, overlap))
            if len(batches) > 1 and len(batches[-1]) <= overlap:
                batches.pop()

            tagged_words = []
            for batch in batches:
                batch_overlap = 0 if batch is batches[-1] else overlap
                joined = " ".join(batch)
                tagged = self._tag(joined)
                assert tagged and tagged[-1]["end"] == len(joined), (
                    f"chunk_size={chunk_size} is too large for {PUNCT_MODEL}: the chunk "
                    f"was truncated by the model's own maximum sequence length"
                )
                char_index = 0
                result_index = 0
                for word in batch[:len(batch) - batch_overlap]:
                    char_index += len(word) + 1
                    label = "0"
                    while result_index < len(tagged) and char_index > tagged[result_index]["end"]:
                        label = tagged[result_index]["entity"]
                        result_index += 1
                    tagged_words.append((word, label))

            assert len(tagged_words) == len(words), (
                f"tagged {len(tagged_words)} words for {len(words)} input words"
            )
            out = ""
            for word, label in tagged_words:
                out += word
                if label == "0":
                    out += " "
                if label in ".,?-:":
                    out += label + " "
            return out.strip()

    def restore_or_keep(model, text):
        if pd.isna(text) or str(text).strip() == "":
            return text
        return model.restore_punctuation(str(text))

    df = pd.read_csv(CSV_PATH, sep='|', names=['file_name', 'transcript'], header=None, quoting=csv.QUOTE_NONE)
    print(f"Loading {PUNCT_MODEL} ({PUNCT_MODEL_SUBDIR}) through onnxruntime...")
    model = OnnxPunctuationModel(PUNCT_MODEL, PUNCT_MODEL_SUBDIR)

    print("Restoring punctuation...")
    df['transcript'] = df['transcript'].apply(lambda text: restore_or_keep(model, text))

    df.to_csv(OUTPUT_PATH, sep='|', index=False, header=False, quoting=csv.QUOTE_NONE)
    print(f"Done! Saved to {OUTPUT_PATH}")


---
### Step 8: Manifest

Writes `manifest.jsonl` and its `.meta.json` sidecar under `WORK_DIR`, following
[manifest.md](../docs/manifest.md). Every filtered donor clip becomes a
pre-conversion row (`id`, `text`, `lang`, `audio`, `duration_s`, `sample_rate`,
`source`, and the `scores` of Gates 1 and 2) with its own `id`; every clip produced by Step 6 becomes a converted
row that adds `identity` (the target voice, from `REF_VOICES_DIR`) and
`parent_id` (the pre-conversion row it was made from) and the `scores` of Gate 3. `text` comes from
`metadata_punct.csv` when Step 7 ran, otherwise from the pre-punctuation
`metadata_filtered.csv`. Each `DIR_FINAL_VC` voice directory also gets its own
LJSpeech-style `metadata.csv`, for tools that expect that format directly.


In [ ]:
import csv
import json
import os
from importlib.metadata import version as _pkg_version

import soundfile as sf

punct_csv = os.path.join(_env("WORK_DIR"), "metadata_punct.csv")
filtered_csv = os.path.join(_env("DIR_FILTERED"), "metadata_filtered.csv")
text_csv = punct_csv if os.path.isfile(punct_csv) else filtered_csv

transcripts = {}
with open(text_csv, newline="", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE):
        if len(row) < 2:
            continue
        fname = row[0] if row[0].endswith(".wav") else row[0] + ".wav"
        transcripts[fname] = row[1]

corpus_name = os.path.basename(os.path.normpath(_env("RAW_AUDIO_SOURCE")))
lang = _env("TTS_LANG", "en-US")
work_dir = _env("WORK_DIR")
filtered_dir = _env("DIR_FILTERED")
final_vc_dir = _env("DIR_FINAL_VC")

# Pre-conversion rows: one per filtered donor clip, each with its own id and
# no "identity" (docs/manifest.md: identity is written by conversion only).
parent_rows = {}


def _row_scores(rec, rename):
    scores = {}
    for key, value in ((rec or {}).get("scores") or {}).items():
        name = rename(key)
        if name:
            scores[name] = value
    return scores


for fname in sorted(f for f in os.listdir(filtered_dir) if f.endswith(".wav")):
    if fname not in transcripts:
        continue
    text = transcripts[fname]
    audio_path = os.path.join(filtered_dir, fname)
    info = sf.info(audio_path)
    stem = os.path.splitext(fname)[0]
    parent_rows[fname] = {
        "id": f"src-{stem}",
        "text": text,
        "lang": lang,
        "audio": os.path.relpath(audio_path, work_dir),
        "duration_s": round(info.duration, 3),
        "sample_rate": info.samplerate,
        "source": {"kind": "corpus", "name": corpus_name},
    }
    # Gate scores: the denoised clip's DNSMOS/NISQA (measured before trimming and
    # normalising), and the transcript check on the cleaned clip.
    parent_scores = _row_scores(CLIPS.get(fname), lambda k: k[:-len("_after")] if k.endswith("_after") else (k if k in ("hyp", "cer") else None))
    if parent_scores:
        parent_rows[fname]["scores"] = parent_scores

if not parent_rows:
    raise RuntimeError(
        f"No transcribed clip found in {filtered_dir} (against {text_csv}); "
        f"manifest.jsonl would have no rows"
    )

manifest_rows = list(parent_rows.values())

# Converted rows: one per (parent clip, target voice), following manifest.md --
# "identity" names the conversion target, "parent_id" the row it was made from;
# "speaker" is left to whichever step names the source speaker (this donor
# corpus does not).
for voice_id in sorted(os.listdir(final_vc_dir)):
    voice_dir = os.path.join(final_vc_dir, voice_id)
    if not os.path.isdir(voice_dir):
        continue

    ljspeech_rows = []
    for fname in sorted(f for f in os.listdir(voice_dir) if f.endswith(".wav")):
        parent = parent_rows.get(fname)
        if parent is None:
            continue
        audio_path = os.path.join(voice_dir, fname)
        info = sf.info(audio_path)
        manifest_rows.append({
            "id": f"{parent['id']}-{voice_id}",
            "parent_id": parent["id"],
            "identity": voice_id,
            "text": parent["text"],
            "lang": parent["lang"],
            "audio": os.path.relpath(audio_path, work_dir),
            "duration_s": round(info.duration, 3),
            "sample_rate": info.samplerate,
            "source": parent["source"],
        })
        converted_scores = _row_scores(CONVERTED.get(f"{voice_id}/{fname}"), lambda k: k)
        if converted_scores:
            manifest_rows[-1]["scores"] = converted_scores
        ljspeech_rows.append(f"{os.path.splitext(fname)[0]}|{parent['text']}")

    with open(os.path.join(voice_dir, "metadata.csv"), "w", encoding="utf-8") as f:
        f.write("\n".join(ljspeech_rows))

converted_rows = [r for r in manifest_rows if r.get("identity")]
if not converted_rows:
    raise RuntimeError(
        "manifest.jsonl would have no voice-converted rows: no converted clip passed gate3_converted "
        "(MIN_SPK_COS, MIN_UTMOS, MAX_CER). See quality_scores.csv and rejected/gate3_converted/."
    )

manifest_path = os.path.join(work_dir, "manifest.jsonl")
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

meta = {
    "notebooks": ["asr2tts"],
    "row_count": len(manifest_rows),
    "instruments": {
        "audiosronnx": {"package": "audiosronnx", "version": _pkg_version("audiosronnx"),
                        "engine": _env("DENOISE_ENGINE", "gtcrn")},
        "voiceclonnx": {"package": "voiceclonnx", "version": _pkg_version("voiceclonnx"),
                        "engine": _env("VC_ENGINE", "chatterbox")},
    },
}
meta["instruments"]["speechonnxmetrics"] = {
    "package": "speechonnxmetrics", "version": _pkg_version("speechonnxmetrics"),
    "scores": "dnsmos_*/nisqa_mos on source rows (denoised clip, before trimming and normalising); "
              "cer on source rows; spk_cos and utmos on converted rows",
}
meta["instruments"]["speakeronnx"] = {
    "package": "speakeronnx", "version": _pkg_version("speakeronnx"), "model": "wespeaker-resnet34",
}
if asr_instrument:
    meta["instruments"]["hyp"] = asr_instrument
else:
    meta["instruments"]["round_trip"] = "skipped (see the gate2_transcript entry in gates)"
meta["gates"] = GATE_LOG
meta["distributions"] = report["distributions"]
with open(manifest_path + ".meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Manifest: {len(parent_rows)} source row(s), {len(converted_rows)} converted row(s) -> {manifest_path}")


In [ ]:
work_dir = _env("WORK_DIR")
kept_clips = sum(v["clips"] for v in report["kept"].values())
kept_min = sum(v["hours"] for v in report["kept"].values()) * 60
print("=" * 78)
print("DONE" + (" (demo run)" if DEMO else ""))
for voice in report["kept"]:
    print(f"  Kept audio     : {os.path.join(_env('DIR_FINAL_VC'), voice)} ({report['kept'][voice]['clips']} clip(s), with metadata.csv)")
print(f"  Rejected audio : {os.path.join(work_dir, 'rejected')}  (one folder per gate; reasons in quality_scores.csv)")
print(f"  Kept in total  : {kept_min:.1f} minutes across {len(report['kept'])} voice(s)")
print(f"  Manifest       : {os.path.join(work_dir, 'manifest.jsonl')} for tts_select, tts_publish and train_vits")
print("  Conversion is stochastic, so a rerun can keep a different set of clips.")
print("  This is far too little to train a voice: train_vits.ipynb needs several hours of GPU training on a")
print("  real dataset, and its own 32-utterance smoke path yields no usable voice. Use hours of speech, not minutes.")
if DEMO:
    print("  For a real Common Voice run set, before the configuration cell:")
    print("    COMMON_VOICE_DIR=<extracted language folder with clips/ and validated.tsv>")
    print("    REF_VOICES_DIR=<folder with one WAV per target voice>   TTS_LANG=<BCP-47 tag, e.g. pt-PT>")
    print("    optional: CV_SPEAKER=auto  CV_MAX_CLIPS=<n>  VC_ENGINE=<engine>  MIN_SPK_COS=<threshold>")
print("=" * 78)

Licensed under Apache-2.0.